In [5]:
import numpy as np
import pandas as pd

df = pd.read_csv("Day16_Student_Wellbeing_Survey.csv")

print("--- TASK 1: DATASET INSPECTION ---")
print("Shape:", df.shape)
print("\nData Types & Info:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nFirst 5 Rows:")
print(df.head())

desc_vars = [
    "Weekly_Study_Hours",
    "Average_Sleep_Hours",
    "Daily_Screen_Time_Hours",
    "Stress_Score",
    "Academic_Readiness_Score",
]

desc_results = []
for col in desc_vars:
  s = df[col]
  mean_val = s.mean()
  median_val = s.median()
  mode_val = s.mode()[0]
  range_val = s.max() - s.min()
  var_val = s.var()
  std_val = s.std()
  q1_val = s.quantile(0.25)
  q3_val = s.quantile(0.75)
  iqr_val = q3_val - q1_val
  cv_val = (std_val / mean_val) * 100

  desc_results.append({
      "Variable": col,
      "Mean": round(mean_val, 3),
      "Median": round(median_val, 3),
      "Mode": round(mode_val, 3),
      "Range": round(range_val, 3),
      "Variance": round(var_val, 3),
      "Std Dev": round(std_val, 3),
      "Q1": round(q1_val, 3),
      "Q3": round(q3_val, 3),
      "IQR": round(iqr_val, 3),
      "CV (%)": round(cv_val, 2),
  })

desc_table = pd.DataFrame(desc_results)
print("\n--- TASK 2: DESCRIPTIVE STATISTICS TABLE ---")
print(desc_table.to_string(index=False))

print("\nVariability Summary:")
print("Greatest Absolute Variability (Std Dev): Academic_Readiness_Score (9.694)")
print(
    "Greatest Relative Variability (CV): Daily_Screen_Time_Hours (41.36%)"
)
outlier_vars = [
    "Weekly_Study_Hours",
    "Daily_Screen_Time_Hours",
    "Commute_Time_Minutes",
    "Monthly_Discretionary_Spending",
]

print("\n--- TASK 3: OUTLIER DETECTION ---")
for col in outlier_vars:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  lb = q1 - 1.5 * iqr
  ub = q3 + 1.5 * iqr
  outliers = df[(df[col] < lb) | (df[col] > ub)][col]

  print(f"\nVariable: {col}")
  print(f"Q1: {q1:.3f}, Q3: {q3:.3f}, IQR: {iqr:.3f}")
  print(f"Lower Bound: {lb:.3f}, Upper Bound: {ub:.3f}")
  print(f"Outlier Count: {len(outliers)}")
  print(f"Outliers: {sorted([round(x, 2) for x in outliers.tolist()])}")

spend_col = "Monthly_Discretionary_Spending"
q1_s = df[spend_col].quantile(0.25)
q3_s = df[spend_col].quantile(0.75)
iqr_s = q3_s - q1_s
lb_s = q1_s - 1.5 * iqr_s
ub_s = q3_s + 1.5 * iqr_s

mean_before = df[spend_col].mean()
median_before = df[spend_col].median()

filtered_spending = df[
    (df[spend_col] >= lb_s) & (df[spend_col] <= ub_s)
][spend_col]
mean_after = filtered_spending.mean()
median_after = filtered_spending.median()

print(f"\nOutlier Impact on {spend_col}:")
print(f"Before Removal -> Mean: {mean_before:.2f}, Median: {median_before:.2f}")
print(f"After Removal  -> Mean: {mean_after:.2f}, Median: {median_after:.2f}")
print(f"Difference     -> Mean: {mean_after - mean_before:.2f}, Median: {median_after - median_before:.2f}")

print("\n--- TASK 4: PROBABILITY ---")
print("Verified column for exercise:", "Exercise_Days_Per_Week")

N = len(df)
event_A = df["Part_Time_Job"] == "Yes"
event_B = df["Stress_Score"] >= 7
event_C = df["Scholarship"] == "Yes"
event_D = df["Exercise_Days_Per_Week"] >= 3

p_A = event_A.sum() / N
p_B = event_B.sum() / N
p_C = event_C.sum() / N
p_D = event_D.sum() / N

p_A_or_B = (event_A | event_B).sum() / N
p_A_and_B = (event_A & event_B).sum() / N

p_A_given_B = (event_A & event_B).sum() / event_B.sum()
p_B_given_A = (event_A & event_B).sum() / event_A.sum()

print(f"P(A)       = {p_A:.4f} ({event_A.sum()}/{N})")
print(f"P(B)       = {p_B:.4f} ({event_B.sum()}/{N})")
print(f"P(C)       = {p_C:.4f} ({event_C.sum()}/{N})")
print(f"P(D)       = {p_D:.4f} ({event_D.sum()}/{N})")
print(f"P(A or B)  = {p_A_or_B:.4f} ({(event_A | event_B).sum()}/{N})")
print(f"P(A and B) = {p_A_and_B:.4f} ({(event_A & event_B).sum()}/{N})")
print(f"P(A|B)     = {p_A_given_B:.4f} ({(event_A & event_B).sum()}/{event_B.sum()})")
print(f"P(B|A)     = {p_B_given_A:.4f} ({(event_A & event_B).sum()}/{event_A.sum()})")

year_overlap = ((df["Year_of_Study"] == 1) & (df["Year_of_Study"] == 4)).sum()
print(f"Students in Year 1 AND Year 4: {year_overlap}")
print(f"Mutually Exclusive: {year_overlap == 0}")

print("\n--- TASK 5: INDEPENDENCE ---")
p_prod = p_A * p_B

print(f"P(A and B)      = {p_A_and_B:.5f}")
print(f"P(A) * P(B)     = {p_prod:.5f}")
print(f"Difference      = {abs(p_A_and_B - p_prod):.5f}")
print(f"Independent     = {np.isclose(p_A_and_B, p_prod)}")

print("\n--- TASK 6: BAYES' THEOREM ---")
event_not_A = ~event_A
p_not_A = event_not_A.sum() / N
p_B_given_not_A = (event_not_A & event_B).sum() / event_not_A.sum()

p_bayes_A_given_B = (p_B_given_A * p_A) / p_B

print(f"P(B | not A)       = {p_B_given_not_A:.5f}")
print(f"P(A | B) via Bayes = {p_bayes_A_given_B:.5f}")
print(f"P(A | B) direct    = {p_A_given_B:.5f}")
print(f"Results Agree      = {np.isclose(p_bayes_A_given_B, p_A_given_B)}")

print("\n--- TASK 7: NORMAL DISTRIBUTION ---")
ars = df["Academic_Readiness_Score"]
mu = ars.mean()
sigma = ars.std()

idx_highest = ars.idxmax()
idx_lowest = ars.idxmin()

highest_stu = df.loc[idx_highest]
lowest_stu = df.loc[idx_lowest]

z_highest = (highest_stu["Academic_Readiness_Score"] - mu) / sigma
z_lowest = (lowest_stu["Academic_Readiness_Score"] - mu) / sigma

within_1 = ((ars >= mu - sigma) & (ars <= mu + sigma)).sum() / N * 100
within_2 = ((ars >= mu - 2 * sigma) & (ars <= mu + 2 * sigma)).sum() / N * 100
within_3 = ((ars >= mu - 3 * sigma) & (ars <= mu + 3 * sigma)).sum() / N * 100

print(f"Mean (mu)           = {mu:.4f}")
print(f"Std Dev (sigma)     = {sigma:.4f}")
print(f"Highest Score: Student {highest_stu['Student_ID']} = {highest_stu['Academic_Readiness_Score']} | Z-score = {z_highest:.4f}")
print(f"Lowest Score:  Student {lowest_stu['Student_ID']} = {lowest_stu['Academic_Readiness_Score']} | Z-score = {z_lowest:.4f}")
print(f"\nEmpirical Rule:")
print(f"Within 1 Std Dev: Theoretical ≈ 68.0% | Actual = {within_1:.2f}%")
print(f"Within 2 Std Dev: Theoretical ≈ 95.0% | Actual = {within_2:.2f}%")
print(f"Within 3 Std Dev: Theoretical ≈ 99.7% | Actual = {within_3:.2f}%")

observations = [
    "1. Academic Readiness Score follows a normal distribution with 69.50%, 95.17%, and 99.83% of values within 1, 2, and 3 standard deviations respectively.",
    "2. Events A (Part-time Job) and B (High Stress) are dependent, as P(A and B) = 0.0517 does not equal P(A) * P(B) = 0.0190.",
    "3. Having a part-time job substantially increases the likelihood of high stress, with P(Stress >= 7 | Job) = 20.39% compared to P(Stress >= 7 | No Job) = 3.13%.",
    "4. Daily Screen Time exhibits the highest relative variability (CV = 41.36%) among behavioral variables, with 18 upper outliers reaching up to 12.4 hours.",
    "5. Monthly Discretionary Spending contains 20 upper outliers, heavily inflating the sample mean by 370.06 while the median shifts by only 95.00 upon removal."
]

print("\n--- : STATISTICAL OBSERVATIONS ---")
for obs in observations:
  print(obs)

--- TASK 1: DATASET INSPECTION ---
Shape: (600, 20)

Data Types & Info:
Student_ID                         object
Age                                 int64
Faculty                            object
Year_of_Study                       int64
City                               object
Accommodation                      object
Scholarship                        object
Part_Time_Job                      object
Internet_Quality                   object
Preferred_Study_Space              object
Weekly_Study_Hours                float64
Average_Sleep_Hours               float64
Daily_Screen_Time_Hours           float64
Exercise_Days_Per_Week              int64
Commute_Time_Minutes              float64
Stress_Score                      float64
Academic_Readiness_Score          float64
Overall_Satisfaction              float64
Monthly_Discretionary_Spending    float64
Social_Activity_Hours_Per_Week    float64
dtype: object

Missing Values:
Student_ID                        0
Age                  